# **CODE 8d: SHAP EXPLAINABILITY & FEATURE ANALYSIS**

Explains the trained XGBoost conviction model using SHAP (TreeExplainer, exact).
Produces thesis-ready charts (vector PDF/SVG + PNG) and tables, grouped feature
contributions (by category and inter-stock dependency), dedicated **High** and
**Medium** class rankings, and a fully-ranked, tunable feature-drop list.

---
## Input files (from Google Drive)
| File | Source |
|------|--------|
| `best_model.pkl` | Code 8b |
| `feature_columns_model.pkl` | Code 8b |
| `categorical_encoders.pkl` | Code 8b |
| `label_mapping.pkl` | Code 8b |
| `feature_importance.csv` | Code 8b (gain importance, for SHAP-vs-gain) |
| `train_data_{SUFFIX}.parquet` / `test*_data_{SUFFIX}.parquet` | Code 8a |
| `feature_reference.csv` | project (category + inter_stock_dependency grouping) |

## Output files (auto-downloaded)
Charts (PDF + SVG + PNG): per-class beeswarm, global importance, inter-stock split,
category contribution, dependence plots, SHAP-vs-gain scatter, High & Medium rankings.
Tables (CSV): importance comparison, category contribution, inter-stock contribution,
High-class ranking, Medium-class ranking, full ranked drop-list.

---
## Key settings (Step 2)
- `SHAP_DATASET` — choose `'train'`, `'test'`, or `'both'`
- `DROP_THRESHOLD_PCT` — percentile cutoff for the drop-list (nothing dropped automatically)
- SHAP is computed on the **FULL** chosen dataset; only plot point-density is subsampled.
---

## **STEP 0: Mount Drive & Install SHAP**

In [1]:
from google.colab import drive
import os
drive.mount('/content/drive')
!pip install shap -q
print("\n✓ Drive mounted, SHAP installed")

Mounted at /content/drive

✓ Drive mounted, SHAP installed


## **STEP 1: Imports**

In [2]:
import pandas as pd
import numpy as np
import pickle
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime

# Thesis-quality plot defaults
plt.rcParams.update({
    'figure.dpi'      : 120,
    'savefig.dpi'     : 300,
    'font.size'       : 11,
    'axes.titlesize'  : 13,
    'axes.titleweight': 'bold',
    'axes.labelsize'  : 11,
    'figure.facecolor': 'white',
    'savefig.bbox'    : 'tight',
})

print(f"✓ Imports ready | SHAP {shap.__version__}")
print(f"  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✓ Imports ready | SHAP 0.52.0
  2026-08-23 15:46:03


## **STEP 2: Configuration**

In [3]:
# ── UPDATE PATH ─────────────────────────────────────────────────────────────
DRIVE_FOLDER = '/content/drive/MyDrive/masters/'
SUFFIX = 'nifty100'

# ── Choose which data to explain ────────────────────────────────────────────
# SHAP_DATASET drives the main rankings/groups/plots (uses 'train' by default).
SHAP_DATASET = 'train'     # 'train' | 'test' | 'both'

# SHAP_EVAL_SET drives the True-vs-False High DISCRIMINATION analysis (Step 12c):
#   'validation' → 2017-2019 slice of the TRAIN parquet (the walk-forward
#                  validation years the model was tuned against)
#   'test'       → Test1 (2020-2022) and Test2 (2023-2025), kept SEPARATE
SHAP_EVAL_SET = 'validation'   # 'validation' | 'test'

# Walk-forward validation window (must match Code 8b folds)
VALIDATION_START = '2017-01-01'
VALIDATION_END   = '2019-12-31'

# ── SHAP COMPUTATION STRATEGY (split: exact numbers, fast plots) ────────────
# The accurate outputs (rankings, group contributions, drop-list) use EXACT
# SHAP via XGBoost's native pred_contribs — GPU-accelerated if available.
# The plots use the (faster) shap library on a subsample, since they can be
# approximate and only display ~PLOT_SUBSAMPLE points anyway.
#
USE_GPU_SHAP   = True      # use GPU for native pred_contribs (needs GPU runtime)
EXACT_MAX_ROWS = 30000     # rows for EXACT pred_contribs (None = full dataset).
#   On GPU, full set is often feasible. On CPU, keep 20-30k. Aggregate rankings
#   & group contributions are stable at 20-30k.
PLOT_APPROX    = True      # plots use approximate shap (fast); set False for exact plots
PLOT_SUBSAMPLE = 10000     # rows used for plot rendering only

# ── Drop-list threshold (percentile of mean|SHAP|; nothing dropped auto) ────
DROP_THRESHOLD_PCT = 15    # Filter 2: features below this percentile FLAGGED
CORR_THRESHOLD     = 0.90  # Filter 1: |corr| >= this groups redundant features


# Inputs
MODEL_FILE    = os.path.join(DRIVE_FOLDER, 'model_combo_068.pkl')
FEATCOLS_FILE = os.path.join(DRIVE_FOLDER, 'feature_columns_model.pkl')
ENCODERS_FILE = os.path.join(DRIVE_FOLDER, 'categorical_encoders.pkl')
LABELMAP_FILE = os.path.join(DRIVE_FOLDER, 'label_mapping.pkl')
GAIN_FILE     = os.path.join(DRIVE_FOLDER, 'feature_importance.csv')
FEATREF_FILE  = os.path.join(DRIVE_FOLDER, 'feature_reference.csv')
TRAIN_FILE    = os.path.join(DRIVE_FOLDER, f'train_data_{SUFFIX}.parquet')
TEST1_FILE    = os.path.join(DRIVE_FOLDER, f'test1_data_{SUFFIX}.parquet')
TEST2_FILE    = os.path.join(DRIVE_FOLDER, f'test2_data_{SUFFIX}.parquet')

# Outputs
OUTPUT_FOLDER = '/content/shap_outputs/'
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print(f"SHAP_DATASET       : {SHAP_DATASET}")
print(f"DROP_THRESHOLD_PCT : {DROP_THRESHOLD_PCT}")
print(f"PLOT_SUBSAMPLE     : {PLOT_SUBSAMPLE:,} (plots only; SHAP computed on full set)")
for f in [MODEL_FILE, FEATCOLS_FILE, ENCODERS_FILE, LABELMAP_FILE, FEATREF_FILE]:
    print(f"  {'✓' if os.path.exists(f) else '✗ MISSING'} {os.path.basename(f)}")

SHAP_DATASET       : train
DROP_THRESHOLD_PCT : 15
PLOT_SUBSAMPLE     : 10,000 (plots only; SHAP computed on full set)
  ✓ model_combo_068.pkl
  ✓ feature_columns_model.pkl
  ✓ categorical_encoders.pkl
  ✓ label_mapping.pkl
  ✓ feature_reference.csv


## **STEP 3: Load Model, Encoders, Reference**

In [4]:
print("-"*80); print("LOADING MODEL & ARTEFACTS"); print("-"*80)

with open(MODEL_FILE, 'rb') as f:    model = pickle.load(f)
with open(FEATCOLS_FILE, 'rb') as f: feature_columns = pickle.load(f)
with open(ENCODERS_FILE, 'rb') as f: label_encoders = pickle.load(f)
with open(LABELMAP_FILE, 'rb') as f: label_mapping = pickle.load(f)
feature_ref = pd.read_csv(FEATREF_FILE)

# Reverse label map: index → name
idx_to_label = {v: k for k, v in label_mapping.items()}
CLASS_ORDER  = [idx_to_label[i] for i in range(len(label_mapping))]
HIGH_IDX     = label_mapping['High']
MEDIUM_IDX   = label_mapping['Medium']

print(f"✓ Model loaded | {len(feature_columns)} features")
print(f"  Class order (by index): {CLASS_ORDER}")
print(f"  High index = {HIGH_IDX}, Medium index = {MEDIUM_IDX}")

# Gain importance (optional, for SHAP-vs-gain comparison)
gain_df = pd.read_csv(GAIN_FILE) if os.path.exists(GAIN_FILE) else None
print(f"  Gain importance: {'loaded' if gain_df is not None else 'not found (skip comparison)'}")

--------------------------------------------------------------------------------
LOADING MODEL & ARTEFACTS
--------------------------------------------------------------------------------
✓ Model loaded | 256 features
  Class order (by index): ['Ignore', 'Low', 'Medium', 'High']
  High index = 3, Medium index = 2
  Gain importance: loaded


## **STEP 4: Load & Encode Data (respecting train/test choice)**

In [5]:
print("-"*80); print(f"LOADING DATA: SHAP_DATASET={SHAP_DATASET} | SHAP_EVAL_SET={SHAP_EVAL_SET}"); print("-"*80)

def _find_col(df, options):
    for c in options:
        if c in df.columns: return c
    return None

def load_encode_full(path, name):
    """Load a parquet, return (X_encoded, labels, dates). Keeps label & date
    alongside the encoded feature matrix for the discrimination analysis."""
    df = pd.read_parquet(path)
    missing = [c for c in feature_columns if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing {len(missing)} features: {missing[:8]}")
    X = df[feature_columns].copy()
    X = X.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
    for col, le in label_encoders.items():
        if col in X.columns:
            known = set(le.classes_)
            X[col] = X[col].fillna('Unknown').astype(str)
            n_unseen = int((~X[col].isin(known)).sum())
            X[col] = X[col].where(X[col].isin(known), le.classes_[0])
            X[col] = le.transform(X[col])
            if n_unseen:
                print(f"    {col}: {n_unseen:,} unseen categories → default")
    # labels + dates (for discrimination analysis); may be absent in some files
    lab_col = _find_col(df, ['conviction_label'])
    dt_col  = _find_col(df, ['date','Date','DATE'])
    labels = df[lab_col].reset_index(drop=True) if lab_col else None
    dates  = pd.to_datetime(df[dt_col]).reset_index(drop=True) if dt_col else None
    print(f"  {name}: {len(X):,} rows"
          f"{' (with labels)' if labels is not None else ' (no labels)'}")
    return X.reset_index(drop=True), labels, dates

# ── Main SHAP datasets (rankings/groups/plots) — unchanged behaviour ─────────
datasets = {}
data_labels = {}   # name → conviction_label series (for discrimination)
if SHAP_DATASET in ('train', 'both'):
    Xtr, ytr, dtr = load_encode_full(TRAIN_FILE, 'train')
    datasets['train'] = Xtr; data_labels['train'] = ytr
if SHAP_DATASET in ('test', 'both'):
    Xt1, yt1, dt1 = load_encode_full(TEST1_FILE, 'test1')
    Xt2, yt2, dt2 = load_encode_full(TEST2_FILE, 'test2')
    datasets['test'] = pd.concat([Xt1, Xt2], ignore_index=True)
    data_labels['test'] = (pd.concat([yt1, yt2], ignore_index=True)
                           if (yt1 is not None and yt2 is not None) else None)
    print(f"  test (combined): {len(datasets['test']):,} rows")

# ── Evaluation datasets for the DISCRIMINATION analysis (Step 12c) ───────────
# Built per SHAP_EVAL_SET; each entry: name → (X_encoded, labels)
eval_sets = {}
if SHAP_EVAL_SET == 'validation':
    Xv, yv, dv = load_encode_full(TRAIN_FILE, 'train(for validation slice)')
    if dv is None:
        raise ValueError("Train parquet has no date column — cannot slice validation years.")
    mask = (dv >= VALIDATION_START) & (dv <= VALIDATION_END)
    eval_sets['Validation_2017_2019'] = (Xv[mask].reset_index(drop=True),
                                         yv[mask].reset_index(drop=True) if yv is not None else None)
    print(f"  Validation slice {VALIDATION_START[:4]}-{VALIDATION_END[:4]}: "
          f"{int(mask.sum()):,} rows")
else:  # 'test' → Test1 and Test2 SEPARATE
    Xt1, yt1, _ = load_encode_full(TEST1_FILE, 'test1')
    Xt2, yt2, _ = load_encode_full(TEST2_FILE, 'test2')
    eval_sets['Test1_2020_2022'] = (Xt1, yt1)
    eval_sets['Test2_2023_2025'] = (Xt2, yt2)

print(f"\n✓ Main datasets: {list(datasets.keys())}")
print(f"✓ Discrimination eval sets: {list(eval_sets.keys())}")

--------------------------------------------------------------------------------
LOADING DATA: SHAP_DATASET=train | SHAP_EVAL_SET=validation
--------------------------------------------------------------------------------
    Sector: 204,764 unseen categories → default
  train: 204,764 rows (with labels)
    Sector: 204,764 unseen categories → default
  train(for validation slice): 204,764 rows (with labels)
  Validation slice 2017-2019: 71,581 rows

✓ Main datasets: ['train']
✓ Discrimination eval sets: ['Validation_2017_2019']


## **STEP 5: Compute SHAP (TreeExplainer, full set)**

In [6]:
print("-"*80); print("COMPUTING SHAP VALUES"); print("-"*80)
import time, xgboost as xgb

booster = model.get_booster()
if USE_GPU_SHAP:
    try:
        booster.set_param({'device': 'cuda'})
        print("  Native pred_contribs will use GPU (device=cuda).")
    except Exception as e:
        print(f"  GPU set failed ({e}); using CPU.")

def exact_shap_native(X, name):
    """EXACT SHAP via XGBoost native pred_contribs (GPU if set). Returns
    array (n_samples, n_features, n_classes)."""
    if EXACT_MAX_ROWS is not None and len(X) > EXACT_MAX_ROWS:
        X = X.sample(EXACT_MAX_ROWS, random_state=42)
        print(f"  {name}: exact SHAP on {len(X):,} rows (sampled)")
    else:
        print(f"  {name}: exact SHAP on {len(X):,} rows (full)")
    t0 = time.time()
    dmat = xgb.DMatrix(X, feature_names=list(X.columns), missing=np.nan)
    contribs = booster.predict(dmat, pred_contribs=True)
    # multiclass → (n_samples, n_classes, n_features+1); drop bias col, reorder
    if contribs.ndim == 3:
        sv = np.transpose(contribs[:, :, :-1], (0, 2, 1))   # → (samples, features, classes)
    else:  # binary/regression edge case
        sv = contribs[:, :-1][:, :, None]
    print(f"    done in {(time.time()-t0)/60:.2f} min | shape {sv.shape}")
    return sv, X.reset_index(drop=True)

# ── EXACT SHAP (for accurate numbers: rankings, groups, drop-list) ──
shap_store = {}     # exact values
shap_X     = {}     # aligned rows
for name, X in datasets.items():
    sv, Xs = exact_shap_native(X, name)
    shap_store[name] = sv
    shap_X[name] = Xs
datasets = shap_X   # downstream numeric steps use these aligned (sampled) rows
print("✓ EXACT SHAP computed for:", list(shap_store.keys()))

# ── APPROX SHAP (for plots only; fast, on a small subsample) ──
PRIMARY = 'test' if 'test' in shap_store else 'train'
X_plot_full = datasets[PRIMARY]
if len(X_plot_full) > PLOT_SUBSAMPLE:
    X_plot = X_plot_full.sample(PLOT_SUBSAMPLE, random_state=7).reset_index(drop=True)
else:
    X_plot = X_plot_full

print(f"\n  Plot SHAP: {'approximate' if PLOT_APPROX else 'exact'} on {len(X_plot):,} rows...")
t0 = time.time()
# GPU predictor does NOT support approximate contributions. The shap library
# uses approximate=PLOT_APPROX, so switch the booster back to CPU for plots
# (the exact pred_contribs above already used GPU and is done).
try:
    booster.set_param({'device': 'cpu'})
except Exception:
    pass
plot_expl = shap.TreeExplainer(model)
sv_plot = plot_expl.shap_values(X_plot, approximate=PLOT_APPROX)
if isinstance(sv_plot, list): sv_plot = np.stack(sv_plot, axis=-1)
elif sv_plot.ndim == 2: sv_plot = sv_plot[:, :, None]
print(f"    plot SHAP done in {(time.time()-t0)/60:.2f} min | shape {sv_plot.shape}")

--------------------------------------------------------------------------------
COMPUTING SHAP VALUES
--------------------------------------------------------------------------------
  Native pred_contribs will use GPU (device=cuda).
  train: exact SHAP on 30,000 rows (sampled)
    done in 1.55 min | shape (30000, 256, 4)
✓ EXACT SHAP computed for: ['train']

  Plot SHAP: approximate on 10,000 rows...
    plot SHAP done in 0.07 min | shape (10000, 256, 4)


## **STEP 6: Helper — save figures as PDF + SVG + PNG**

In [7]:
def save_fig(fig, basename):
    """Save a figure in vector (PDF, SVG) and raster (PNG) formats."""
    paths = []
    for ext in ['pdf', 'svg', 'png']:
        p = os.path.join(OUTPUT_FOLDER, f"{basename}.{ext}")
        fig.savefig(p, format=ext); paths.append(p)
    plt.close(fig); return paths

def mean_abs_shap(sv, class_idx=None):
    """Mean |SHAP| per feature. class_idx=None → averaged across all classes."""
    if class_idx is None:
        return np.abs(sv).mean(axis=(0, 2))
    return np.abs(sv[:, :, class_idx]).mean(axis=0)

# EXACT values drive all NUMERIC outputs (rankings, groups, drop-list)
X_primary  = datasets[PRIMARY]
sv_primary = shap_store[PRIMARY]
print(f"Numbers use EXACT SHAP: {PRIMARY} ({len(X_primary):,} rows)")

# APPROX values drive PLOTS only
X_sub  = X_plot
sv_sub = sv_plot
print(f"Plots use {'approx' if PLOT_APPROX else 'exact'} SHAP ({len(X_sub):,} rows)")

Numbers use EXACT SHAP: train (30,000 rows)
Plots use approx SHAP (10,000 rows)


## **STEP 7: Per-Class Beeswarm Plots (Ignore / Low / Medium / High)**

In [8]:
print("Generating per-class beeswarm plots...")
saved_figs = []
for cls_name in CLASS_ORDER:
    ci = label_mapping[cls_name]
    fig = plt.figure()
    shap.summary_plot(sv_sub[:, :, ci], X_sub, feature_names=feature_columns,
                      max_display=20, show=False)
    plt.title(f"SHAP feature impact on '{cls_name}' conviction", pad=14)
    fig = plt.gcf()
    saved_figs += save_fig(fig, f"shap_beeswarm_{cls_name}")
    print(f"  ✓ {cls_name}")
print("  Beeswarm plots saved (PDF/SVG/PNG)")

Generating per-class beeswarm plots...
  ✓ Ignore
  ✓ Low
  ✓ Medium
  ✓ High
  Beeswarm plots saved (PDF/SVG/PNG)


## **STEP 8: Dedicated HIGH and MEDIUM Class Rankings (charts + tables)**

In [9]:
print("Building dedicated High & Medium rankings...")

def class_ranking_df(class_name):
    ci = label_mapping[class_name]
    vals = mean_abs_shap(sv_primary, ci)
    d = pd.DataFrame({'feature': feature_columns, 'mean_abs_shap': vals})
    d = d.merge(feature_ref[['feature_name','category','inter_stock_dependency']],
                left_on='feature', right_on='feature_name', how='left').drop(columns='feature_name')
    d = d.sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
    d.insert(0, 'rank', d.index + 1)
    return d

for cls in ['High', 'Medium']:
    rk = class_ranking_df(cls)
    rk.to_csv(os.path.join(OUTPUT_FOLDER, f'shap_ranking_{cls}.csv'), index=False)

    top = rk.head(20).iloc[::-1]
    colors = top['inter_stock_dependency'].map({'Yes':'#2E5395','No':'#9AB0D6'}).fillna('#cccccc')
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.barh(top['feature'], top['mean_abs_shap'], color=colors)
    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title(f"Top 20 drivers of '{cls}' conviction\n(blue = inter-stock, light = standalone)")
    saved_figs.extend(save_fig(fig, f"shap_ranking_{cls}_top20"))
    print(f"  ✓ {cls}: chart + shap_ranking_{cls}.csv")

Building dedicated High & Medium rankings...
  ✓ High: chart + shap_ranking_High.csv
  ✓ Medium: chart + shap_ranking_Medium.csv


## **STEP 9: Global Importance + SHAP-vs-Gain Comparison**

In [10]:
print("Global importance & SHAP-vs-gain...")

overall = pd.DataFrame({
    'feature': feature_columns,
    'mean_abs_shap_overall': mean_abs_shap(sv_primary, None),
})
for cls in CLASS_ORDER:
    overall[f'shap_{cls}'] = mean_abs_shap(sv_primary, label_mapping[cls])
overall = overall.merge(
    feature_ref[['feature_name','category','inter_stock_dependency']],
    left_on='feature', right_on='feature_name', how='left').drop(columns='feature_name')
overall = overall.sort_values('mean_abs_shap_overall', ascending=False).reset_index(drop=True)
overall.insert(0, 'shap_rank', overall.index + 1)

# Global bar (top 20)
top = overall.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top['feature'], top['mean_abs_shap_overall'], color='#2E5395')
ax.set_xlabel('Mean |SHAP value| (avg across classes)')
ax.set_title('Top 20 features by overall SHAP importance')
saved_figs.extend(save_fig(fig, 'shap_global_top20'))

# SHAP vs gain scatter
if gain_df is not None:
    gcol = 'importance' if 'importance' in gain_df.columns else gain_df.columns[1]
    cmp = overall.merge(gain_df.rename(columns={gain_df.columns[0]:'feature', gcol:'gain'})[['feature','gain']],
                        on='feature', how='left')
    cmp['gain'] = cmp['gain'].fillna(0)
    cmp['gain_rank'] = cmp['gain'].rank(ascending=False).astype(int)
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(cmp['gain_rank'], cmp['shap_rank'], alpha=0.5, color='#2E5395')
    ax.set_xlabel('Gain importance rank'); ax.set_ylabel('SHAP importance rank')
    ax.set_title('SHAP vs Gain importance rank\n(points off the diagonal: methods disagree)')
    ax.plot([1, len(cmp)], [1, len(cmp)], 'r--', lw=1, alpha=0.6)
    saved_figs.extend(save_fig(fig, 'shap_vs_gain_scatter'))
    cmp[['shap_rank','feature','mean_abs_shap_overall','gain','gain_rank']].to_csv(
        os.path.join(OUTPUT_FOLDER, 'importance_comparison.csv'), index=False)
    print("  ✓ SHAP-vs-gain scatter + importance_comparison.csv")

overall.to_csv(os.path.join(OUTPUT_FOLDER, 'shap_importance_overall.csv'), index=False)
print("  ✓ shap_importance_overall.csv")

Global importance & SHAP-vs-gain...
  ✓ SHAP-vs-gain scatter + importance_comparison.csv
  ✓ shap_importance_overall.csv


## **STEP 10: Grouped Contributions — by `category` and `inter_stock_dependency`**

In [11]:
print("Grouped SHAP contributions...")

# Per-feature mean|SHAP| per class, joined to reference groups
feat_class = pd.DataFrame({'feature': feature_columns})
for cls in CLASS_ORDER:
    feat_class[cls] = mean_abs_shap(sv_primary, label_mapping[cls])
feat_class['Overall'] = mean_abs_shap(sv_primary, None)
feat_class = feat_class.merge(
    feature_ref[['feature_name','category','inter_stock_dependency']],
    left_on='feature', right_on='feature_name', how='left').drop(columns='feature_name')

val_cols = CLASS_ORDER + ['Overall']

# ── By category ──
by_cat = feat_class.groupby('category')[val_cols].sum().sort_values('Overall', ascending=False)
by_cat['pct_of_total'] = (by_cat['Overall'] / by_cat['Overall'].sum() * 100).round(2)
by_cat.to_csv(os.path.join(OUTPUT_FOLDER, 'shap_group_by_category.csv'))

fig, ax = plt.subplots(figsize=(9, 8))
cat_sorted = by_cat.sort_values('Overall')
ax.barh(cat_sorted.index, cat_sorted['Overall'], color='#2E5395')
ax.set_xlabel('Summed mean |SHAP| (overall)')
ax.set_title('Feature-category contribution to predictions')
saved_figs.extend(save_fig(fig, 'shap_group_by_category'))

# Stacked by class (top 12 categories)
topcats = by_cat.head(12).index
fig, ax = plt.subplots(figsize=(10, 7))
bottom = np.zeros(len(topcats))
palette = ['#9AB0D6', '#5B7FB9', '#2E5395', '#16306b']
for i, cls in enumerate(CLASS_ORDER):
    ax.barh(topcats, by_cat.loc[topcats, cls], left=bottom, label=cls, color=palette[i])
    bottom += by_cat.loc[topcats, cls].values
ax.invert_yaxis(); ax.set_xlabel('Summed mean |SHAP|'); ax.legend(title='Class')
ax.set_title('Top categories: contribution split by conviction class')
saved_figs.extend(save_fig(fig, 'shap_group_by_category_stacked'))

# ── By inter-stock dependency ──
by_is = feat_class.groupby('inter_stock_dependency')[val_cols].sum()
by_is['pct_of_total'] = (by_is['Overall'] / by_is['Overall'].sum() * 100).round(2)
by_is.to_csv(os.path.join(OUTPUT_FOLDER, 'shap_group_by_interstock.csv'))

fig, ax = plt.subplots(figsize=(8, 6))
x = np.arange(len(CLASS_ORDER)); w = 0.35
for j, grp in enumerate(by_is.index):
    ax.bar(x + (j-0.5)*w, [by_is.loc[grp, c] for c in CLASS_ORDER], w,
           label=f"inter-stock={grp}", color=['#2E5395','#9AB0D6'][j % 2])
ax.set_xticks(x); ax.set_xticklabels(CLASS_ORDER)
ax.set_ylabel('Summed mean |SHAP|'); ax.legend()
ax.set_title('Inter-stock vs standalone feature contribution, by class')
saved_figs.extend(save_fig(fig, 'shap_group_by_interstock'))

print("  ✓ category & inter-stock group CSVs + charts")
print("\n  Inter-stock contribution share:")
for grp in by_is.index:
    print(f"    {grp:4s}: {by_is.loc[grp,'pct_of_total']:.1f}% of total |SHAP|")

Grouped SHAP contributions...
  ✓ category & inter-stock group CSVs + charts

  Inter-stock contribution share:
    No  : 42.0% of total |SHAP|
    Yes : 58.0% of total |SHAP|


## **STEP 11: Dependence Plots (top features)**

In [12]:
print("Dependence plots for top features...")
top_feats = overall.head(5)['feature'].tolist()
for feat in top_feats:
    fi = feature_columns.index(feat)
    fig = plt.figure()
    # use High class for dependence (your primary question)
    shap.dependence_plot(fi, sv_sub[:, :, HIGH_IDX], X_sub,
                         feature_names=feature_columns, show=False)
    plt.title(f"Dependence: {feat} → 'High' conviction")
    saved_figs.extend(save_fig(plt.gcf(), f"shap_dependence_{feat}"))
    print(f"  ✓ {feat}")

Dependence plots for top features...
  ✓ cluster_num_stocks
  ✓ commodity_return_252d
  ✓ commodity_volatility_252d
  ✓ commodity_min_return_60d
  ✓ Value_Traded_MA20_Log


## **STEP 12: Full Ranked Feature List + Tunable Drop Recommendation**

In [13]:
print("-"*80); print("FULL RANKED FEATURE LIST + DROP RECOMMENDATION"); print("-"*80)

rank = overall.copy()
# attach gain if available
if gain_df is not None:
    gcol = 'importance' if 'importance' in gain_df.columns else gain_df.columns[1]
    g = gain_df.rename(columns={gain_df.columns[0]:'feature', gcol:'gain'})[['feature','gain']]
    rank = rank.merge(g, on='feature', how='left')
    rank['gain'] = rank['gain'].fillna(0)
    rank['gain_rank'] = rank['gain'].rank(ascending=False, method='min').astype(int)
else:
    rank['gain'] = np.nan; rank['gain_rank'] = np.nan

# Threshold on overall mean|SHAP|
cutoff = np.percentile(rank['mean_abs_shap_overall'], DROP_THRESHOLD_PCT)

def reason(row):
    reasons = []
    if row['mean_abs_shap_overall'] <= cutoff:
        reasons.append(f'low SHAP (<= {DROP_THRESHOLD_PCT}th pct)')
    if not np.isnan(row['gain']) and row['gain'] == 0:
        reasons.append('zero gain (never split)')
    if not np.isnan(row['gain_rank']) and row['shap_rank'] > len(rank)*0.85 and row['gain_rank'] > len(rank)*0.85:
        reasons.append('low SHAP AND low gain')
    return '; '.join(reasons) if reasons else 'keep'

rank['drop_reason'] = rank.apply(reason, axis=1)
rank['flagged_for_drop'] = rank['drop_reason'] != 'keep'

cols = ['shap_rank','feature','category','inter_stock_dependency',
        'mean_abs_shap_overall'] + [f'shap_{c}' for c in CLASS_ORDER] + \
       ['gain','gain_rank','drop_reason','flagged_for_drop']
rank_out = rank[[c for c in cols if c in rank.columns]]
rank_out.to_csv(os.path.join(OUTPUT_FOLDER, 'feature_ranking_full.csv'), index=False)

n_flag = int(rank['flagged_for_drop'].sum())
print(f"Cutoff (mean|SHAP| <= {DROP_THRESHOLD_PCT}th pct): {cutoff:.6f}")
print(f"Features flagged for dropping: {n_flag} / {len(rank)}")
print(f"\nALL features ranked (decreasing importance) → feature_ranking_full.csv")
print("\nBottom 15 (drop candidates):")
print(rank_out.tail(15)[['shap_rank','feature','category','mean_abs_shap_overall','drop_reason']].to_string(index=False))
print("\nTo prune: copy flagged feature names into features_to_prune.csv (column 'feature_name'),")
print("set PRUNE_FILE in Code 8b, re-run, and compare performance.")
print("Adjust DROP_THRESHOLD_PCT in Step 2 to widen/narrow the flag list.")

--------------------------------------------------------------------------------
FULL RANKED FEATURE LIST + DROP RECOMMENDATION
--------------------------------------------------------------------------------
Cutoff (mean|SHAP| <= 15th pct): 0.009442
Features flagged for dropping: 39 / 256

ALL features ranked (decreasing importance) → feature_ranking_full.csv

Bottom 15 (drop candidates):
 shap_rank                        feature          category  mean_abs_shap_overall                                   drop_reason
       242       commodity_volatility_20d Commodity Returns               0.006017                        low SHAP (<= 15th pct)
       243         Higher_Highs_Count_10d             HH/LL               0.005356                        low SHAP (<= 15th pct)
       244                 UpDays_Pct_10d           Up-Days               0.005296                        low SHAP (<= 15th pct)
       245                      DayOfWeek              Date               0.004388 low SHAP

## **STEP 12b: Filter 1 — Redundancy Pruning + Combined Recommendation**

Two-filter feature pruning:
- **Filter 1 (redundancy):** within groups of highly-correlated features (|corr| ≥ `CORR_THRESHOLD`), keep the highest mean-|SHAP| feature, flag the rest as redundant. Targets near-duplicate features (e.g. the many cluster/commodity variants).
- **Filter 2 (weak):** features weak on BOTH SHAP and gain (from Step 12).

Outputs a combined `features_to_prune_recommended.csv` ready for Code 8b's `PRUNE_FILE`.

In [14]:
print("-"*80)
print("FILTER 1: REDUNDANCY PRUNING (correlation-based)")
print("-"*80)

# Use the SHAP dataset (already loaded & encoded) for correlation
X_corr = datasets[PRIMARY]
print(f"Computing correlation matrix on {len(X_corr):,} rows × {X_corr.shape[1]} features...")
corr = X_corr.corr().abs()

# Overall mean|SHAP| per feature (from Step 9 `overall`)
shap_imp = overall.set_index('feature')['mean_abs_shap_overall']

# Greedy: walk features by DESCENDING SHAP; each unclaimed feature becomes a
# cluster representative and claims its still-unclaimed correlated neighbors.
order = shap_imp.sort_values(ascending=False).index.tolist()
claimed = set()
redundant_rows = []
representatives = []
for f in order:
    if f in claimed or f not in corr.index:
        continue
    representatives.append(f)
    claimed.add(f)
    neighbors = [g for g in corr.index
                 if g != f and g not in claimed and corr.loc[f, g] >= CORR_THRESHOLD]
    for g in neighbors:
        redundant_rows.append({
            'feature': g,
            'redundant_with': f,
            'abs_corr': round(float(corr.loc[f, g]), 4),
            'feature_shap': round(float(shap_imp.get(g, 0)), 6),
            'kept_feature_shap': round(float(shap_imp.get(f, 0)), 6),
        })
        claimed.add(g)

redundancy_df = pd.DataFrame(redundant_rows)
redundancy_df.to_csv(os.path.join(OUTPUT_FOLDER, 'redundancy_prune_list.csv'), index=False)

print(f"  Correlation threshold: {CORR_THRESHOLD}")
print(f"  Cluster representatives kept : {len(representatives)}")
print(f"  Redundant features flagged   : {len(redundancy_df)}")
if len(redundancy_df):
    print("\n  Sample redundant features (dropped vs kept):")
    print(redundancy_df.head(12).to_string(index=False))
print("\n  ✓ redundancy_prune_list.csv saved")

# ── Combine Filter 1 (redundancy) + Filter 2 (weak SHAP+gain) ────────────────
print("\n" + "-"*80)
print("COMBINED RECOMMENDATION (Filter 1 ∪ Filter 2)")
print("-"*80)

redundant_set = set(redundancy_df['feature']) if len(redundancy_df) else set()
# Filter 2 weak set: flagged in Step 12 ranking
weak_set = set(rank.loc[rank['flagged_for_drop'], 'feature']) if 'rank' in dir() else set()

combined = []
for feat in feature_columns:
    reasons = []
    if feat in redundant_set:
        match = redundancy_df.loc[redundancy_df['feature']==feat, 'redundant_with'].iloc[0]
        reasons.append(f'redundant with {match}')
    if feat in weak_set:
        reasons.append('weak (low SHAP+gain)')
    if reasons:
        combined.append({
            'feature': feat,
            'mean_abs_shap': round(float(shap_imp.get(feat, 0)), 6),
            'filter1_redundant': feat in redundant_set,
            'filter2_weak': feat in weak_set,
            'prune_reason': '; '.join(reasons),
        })

combined_df = pd.DataFrame(combined).sort_values('mean_abs_shap')
# The file Code 8b reads needs a 'feature_name' column
prune_out = combined_df.rename(columns={'feature': 'feature_name'})
prune_out.to_csv(os.path.join(OUTPUT_FOLDER, 'features_to_prune_recommended.csv'), index=False)

print(f"  Filter 1 (redundant): {len(redundant_set)}")
print(f"  Filter 2 (weak)     : {len(weak_set)}")
print(f"  Combined unique     : {len(combined_df)}  ({len(combined_df)/len(feature_columns)*100:.1f}% of features)")
print(f"  Features remaining if pruned: {len(feature_columns) - len(combined_df)}")
print("\n  ✓ features_to_prune_recommended.csv saved (column 'feature_name')")
print("\n  NEXT STEP: set PRUNE_FILE to this CSV in Code 8b, re-run, and confirm")
print("  walk-forward CV utility is maintained or improved. Validate before keeping.")
print()

--------------------------------------------------------------------------------
FILTER 1: REDUNDANCY PRUNING (correlation-based)
--------------------------------------------------------------------------------
Computing correlation matrix on 30,000 rows × 256 features...
  Correlation threshold: 0.9
  Cluster representatives kept : 219
  Redundant features flagged   : 37

  Sample redundant features (dropped vs kept):
                   feature            redundant_with  abs_corr  feature_shap  kept_feature_shap
 commodity_volatility_120d commodity_volatility_252d    0.9262      0.034732           0.102171
  commodity_volatility_60d  commodity_min_return_60d    0.9341      0.008885           0.101951
commodity_max_drawdown_60d  commodity_min_return_60d    0.9108      0.034333           0.101951
   cluster_volatility_120d   cluster_volatility_252d    0.9348      0.046535           0.065075
           Volatility_120d           Volatility_252d    0.9014      0.045150           0.059939
 

## **STEP 12c: True-vs-False HIGH Discrimination (feature-level)**

For each feature, compares its mean SHAP-toward-High on rows the model **correctly**
calls High (actual=High, pred=High) versus rows it **wrongly** calls High
(actual=Ignore→High, actual=Low→High).

**Discrimination score** = mean SHAP-High(True-High) − mean SHAP-High(False-High).
- High positive → feature drives *correct* High calls → **keep**
- Negative / low → feature drives *wrong* High calls (Ignore/Low→High) → **prune candidate**

Runs on `SHAP_EVAL_SET` (validation 2017-2019, or Test1 & Test2 separately).
Row counts per bucket are printed — treat buckets with <30 rows as low-confidence.

In [15]:
print("-"*80)
print(f"TRUE-vs-FALSE HIGH DISCRIMINATION  (eval = {SHAP_EVAL_SET})")
print("-"*80)
import xgboost as xgb

# native exact pred_contribs for an eval set → (n, features, classes)
def exact_contribs(X):
    try: booster.set_param({'device': 'cuda' if USE_GPU_SHAP else 'cpu'})
    except Exception: pass
    dmat = xgb.DMatrix(X, feature_names=list(X.columns), missing=np.nan)
    contribs = booster.predict(dmat, pred_contribs=True)
    if contribs.ndim == 3:
        return np.transpose(contribs[:, :, :-1], (0, 2, 1))
    return contribs[:, :-1][:, :, None]

disc_results = {}
for set_name, (Xe, ye) in eval_sets.items():
    print(f"\n=== {set_name} ({len(Xe):,} rows) ===")
    if ye is None:
        print("  No labels available for this set — skipping discrimination.")
        continue

    # Predictions on the SAME rows
    preds = model.predict(Xe)               # numeric class indices
    pred_names = pd.Series(preds).map(idx_to_label).values
    actual_names = ye.values

    # Exact SHAP for the High class on these rows
    sv_e = exact_contribs(Xe)               # (n, features, classes)
    shap_high = sv_e[:, :, HIGH_IDX]        # (n, features) — push toward High

    # Row masks
    m_true_high  = (actual_names == 'High')   & (pred_names == 'High')
    m_ig_to_high = (actual_names == 'Ignore') & (pred_names == 'High')
    m_lo_to_high = (actual_names == 'Low')    & (pred_names == 'High')
    m_false_high = m_ig_to_high | m_lo_to_high

    n_tp, n_ig, n_lo = int(m_true_high.sum()), int(m_ig_to_high.sum()), int(m_lo_to_high.sum())
    print(f"  Rows: True-High={n_tp:,} | Ignore→High={n_ig:,} | Low→High={n_lo:,} "
          f"| False-High={n_ig+n_lo:,}")
    if n_tp < 30 or (n_ig + n_lo) < 30:
        print(f"  ⚠️  Small bucket(s) (<30 rows) — discrimination scores low-confidence here.")

    def safe_mean(arr, mask):
        return arr[mask].mean(axis=0) if mask.sum() > 0 else np.zeros(arr.shape[1])

    mean_tp     = safe_mean(shap_high, m_true_high)
    mean_ig     = safe_mean(shap_high, m_ig_to_high)
    mean_lo     = safe_mean(shap_high, m_lo_to_high)
    mean_false  = safe_mean(shap_high, m_false_high)

    df = pd.DataFrame({
        'feature'                  : feature_columns,
        'shapHigh_trueHigh'        : np.round(mean_tp, 6),
        'shapHigh_IgnoreToHigh'    : np.round(mean_ig, 6),
        'shapHigh_LowToHigh'       : np.round(mean_lo, 6),
        'shapHigh_falseHigh'       : np.round(mean_false, 6),
        'discrimination_score'     : np.round(mean_tp - mean_false, 6),
    })
    df = df.merge(feature_ref[['feature_name','category','inter_stock_dependency']],
                  left_on='feature', right_on='feature_name', how='left').drop(columns='feature_name')
    df = df.sort_values('discrimination_score')   # worst (prune) first
    df.insert(0, 'prune_rank', range(1, len(df)+1))

    fname = f'high_discrimination_{set_name}.csv'
    df.to_csv(os.path.join(OUTPUT_FOLDER, fname), index=False)
    disc_results[set_name] = df

    print(f"  ✓ saved {fname}")
    print(f"\n  Worst 12 (drive FALSE-High more than true-High → prune candidates):")
    print(df.head(12)[['feature','category','shapHigh_trueHigh',
                       'shapHigh_falseHigh','discrimination_score']].to_string(index=False))
    print(f"\n  Best 8 (clean True-High drivers → keep):")
    print(df.tail(8)[['feature','shapHigh_trueHigh','shapHigh_falseHigh',
                      'discrimination_score']].to_string(index=False))

# If two test sets, flag features that are prune candidates in BOTH (consistent)
if len(disc_results) == 2:
    names = list(disc_results.keys())
    a, b = disc_results[names[0]], disc_results[names[1]]
    bottom_a = set(a.head(int(len(a)*0.20))['feature'])
    bottom_b = set(b.head(int(len(b)*0.20))['feature'])
    consistent = sorted(bottom_a & bottom_b)
    cons_df = pd.DataFrame({'feature_name': consistent})
    cons_df.to_csv(os.path.join(OUTPUT_FOLDER, 'high_discrimination_consistent_prune.csv'), index=False)
    print(f"\n  ✓ {len(consistent)} features are bottom-20% false-High drivers in BOTH "
          f"{names[0]} & {names[1]}")
    print(f"    → high_discrimination_consistent_prune.csv (most defensible prune set)")
print()

--------------------------------------------------------------------------------
TRUE-vs-FALSE HIGH DISCRIMINATION  (eval = validation)
--------------------------------------------------------------------------------

=== Validation_2017_2019 (71,581 rows) ===
  Rows: True-High=8,582 | Ignore→High=5 | Low→High=5 | False-High=10
  ⚠️  Small bucket(s) (<30 rows) — discrimination scores low-confidence here.
  ✓ saved high_discrimination_Validation_2017_2019.csv

  Worst 12 (drive FALSE-High more than true-High → prune candidates):
                   feature          category  shapHigh_trueHigh  shapHigh_falseHigh  discrimination_score
                   cluster      Cluster Info           0.029366            0.142457             -0.113092
        cluster_num_stocks   Cluster Returns          -0.011689            0.075432             -0.087121
 commodity_volatility_252d Commodity Returns           0.218278            0.296482             -0.078203
    cluster_min_return_60d   Cluster Retur

## **STEP 13: Auto-Download All Outputs**

In [16]:
print("="*80); print("AUTO-DOWNLOADING OUTPUTS"); print("="*80)
from google.colab import files
import time

# Browsers throttle/drop rapid-fire automatic downloads (often after ~10 in a
# burst). A short delay between each call prevents files from being silently
# dropped. If you still see fewer files than expected, check your browser's
# download-permission prompt (it may be waiting for a one-time "Allow") and/or
# re-run this cell — files.download() is safe to call again for the same file.
all_files = sorted(f for f in os.listdir(OUTPUT_FOLDER)
                   if os.path.isfile(os.path.join(OUTPUT_FOLDER, f)))
print(f"Downloading {len(all_files)} files (with delay to avoid browser throttling)...")

for i, fname in enumerate(all_files):
    fp = os.path.join(OUTPUT_FOLDER, fname)
    try:
        files.download(fp)
        print(f"  ✓ ({i+1}/{len(all_files)}) {fname}")
    except Exception as e:
        print(f"  ✗ {fname}: {e}")
    time.sleep(1.5)   # delay prevents the browser from dropping later downloads

print(f"\nIf fewer than {len(all_files)} files appear in your Downloads folder,")
print(f"check for a browser permission prompt.")

# ── MORE RELIABLE ALTERNATIVE: download everything as ONE zip ──────────────
# A single zip avoids browser download-throttling entirely. Recommended when
# you have many output files (as here). Uncomment to use instead of the loop
# above (you can run this even after the loop, as a safety net):
#
import shutil
zip_path = shutil.make_archive('/content/shap_outputs_zip', 'zip', OUTPUT_FOLDER)
files.download(zip_path)
print(f"✓ Downloaded all outputs as one zip: {zip_path}")
print("\n" + "="*80); print("CODE 8d COMPLETE"); print("="*80)
print("Vector charts (PDF/SVG) + PNG and CSV tables ready for the thesis.")

AUTO-DOWNLOADING OUTPUTS


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ✓ (1/58) feature_ranking_full.csv


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

  ✓ (58/58) shap_vs_gain_scatter.svg

If fewer than 58 files appear in your Downloads folder,
check for a browser permission prompt.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Downloaded all outputs as one zip: /content/shap_outputs_zip.zip

CODE 8d COMPLETE
Vector charts (PDF/SVG) + PNG and CSV tables ready for the thesis.
